# SOFR prediction (Jupyter / DataZone)

Equivalente a:

```bash
python run_dz.py prediction --once --pull-history
```

**Datos en disco** (carpeta del repo):
- `sofr_live.csv` — panel BBG acumulado (`curve_config.LIVE_STORE_PATH`)
- `predictions_latest.csv` — última tabla de predicción

Antes de correr: `git pull origin main` en el clone. Ajusta tenors/historia en `curve_config.py`.

In [ ]:
import importlib
import logging
from IPython.display import display
import os
import sys
from pathlib import Path

import pandas as pd

# Silenciar logs ruidosos de bbgapi en Jupyter
logging.disable(logging.WARNING)
logging.getLogger().setLevel(logging.ERROR)
for _n in list(logging.root.manager.loggerDict):
    lg = logging.getLogger(_n)
    if any(k in str(_n).lower() for k in ("bbg", "blp", "dz", "datazone", "activity", "ms")):
        lg.disabled = True
        lg.handlers = []

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

# Repo: cwd si ya abriste el notebook desde fwd_screener_v1; si no, ruta DataZone
_DZ = Path("/v/region/na/appl/datazone/notebooks2/data/users/arraipau/fwd_screener_v1")
REPO = Path.cwd() if (Path.cwd() / "run_prediction.py").exists() else _DZ
if not (REPO / "run_prediction.py").exists():
    raise FileNotFoundError(
        "No encuentro run_prediction.py en %s ni en %s — clona el repo o cambia _DZ."
        % (Path.cwd(), _DZ)
    )

os.chdir(REPO)
sys.path = [str(REPO)] + [p for p in sys.path if str(Path(p).resolve()) != str(REPO.resolve())]

print("REPO:", REPO.resolve())
print("python:", sys.executable)
print("pandas:", pd.__version__)

In [ ]:
# Paquetes DataZone (urllib3, certifi, bbgapi, …) — mismo bootstrap que run_dz.py
import run_dz

importlib.reload(run_dz)
run_dz.bootstrap_for_repo(str(REPO))

In [ ]:
# Opciones (equivalente CLI)
REFRESH = True           # False → --no-refresh (solo CSV en disco)
PULL_HISTORY = True      # True → --pull-history (reconstruye sofr_live.csv vía bdib)

import curve_config
from curve_config import CHOSEN_MODELS, LIVE_STORE_PATH, MAX_HISTORY_DAYS, BBG_INTERVAL_MINUTES

print("LIVE_STORE:", (REPO / LIVE_STORE_PATH).resolve())
print("history ~%d days, interval %dm, models:" % (MAX_HISTORY_DAYS, BBG_INTERVAL_MINUTES),
      [m.name for m in CHOSEN_MODELS])

In [ ]:
import data
import run_prediction

importlib.reload(data)
importlib.reload(run_prediction)

table = run_prediction.run_once(
    refresh=REFRESH,
    force_history=PULL_HISTORY,
)

cols = [
    c
    for c in (
        "model",
        "structure",
        "features",
        "nivel",
        "y_hat_now",
        "residual",
        "y_hat_next",
        "equation",
    )
    if c in table.columns
]
display(table[cols])

In [ ]:
# Comprobar store en disco
store = REPO / LIVE_STORE_PATH
pred_csv = REPO / "predictions_latest.csv"

for p in (store, pred_csv):
    if p.exists():
        print("%s  size=%s  mtime=%s" % (p.name, p.stat().st_size, pd.Timestamp(p.stat().st_mtime, unit="s")))
    else:
        print("missing:", p)

if store.exists():
    live = pd.read_csv(store, parse_dates=["time"], index_col="time")
    print("sofr_live rows:", len(live), "last:", live.index[-1])
    display(live.iloc[-1:].T.head(20))

## Opcional: smoke `bdib` (1 ticker)

Si todos los tickers fallan con `'//blp/refdata'`, el problema es sesión BBG / servicio intraday, no el notebook.

In [ ]:
from datetime import datetime, timedelta

from data import _quiet_bbg_logs, _bquery

_quiet_bbg_logs()
bq = _bquery()
end = datetime.now().replace(microsecond=0)
start = end - timedelta(days=5)
fmt = "%Y-%m-%dT%H:%M:%S"

bars = bq.bdib(
    security="USOSFR10 BGN Curncy",
    event_type="TRADE",
    interval=int(BBG_INTERVAL_MINUTES),
    start_datetime=start.strftime(fmt),
    end_datetime=end.strftime(fmt),
)
print("shape:", getattr(bars, "shape", None), "cols:", list(bars.columns) if bars is not None else None)
bars.head() if bars is not None else "bdib returned None"